In [ ]:
!pip install pyxdf

import pyxdf
import numpy as np
import os

DATA_DIR = "/kaggle/input/datasets/tanishguptttaaa/eggdatasett"

DOWNSAMPLE = 2
window_size = 100
step_size = 100

subjects = [i for i in range(1, 45) if i not in [4, 11]]

all_samples = []
all_labels = []

marker_global_map = {}
current_label_id = 0

for sub in subjects:
    file_path = os.path.join(DATA_DIR, f"S{sub:02d}.xdf")
    print(f"\nProcessing {file_path}")

    streams, _ = pyxdf.load_xdf(file_path)

    EEG_stream, ET_stream, MRK_stream = None, None, None

    for s in streams:
        name = s['info']['name'][0]
        typ = s['info']['type'][0]

        if typ == 'EEG':
            EEG_stream = s
        elif typ == 'Eyetracker':
            ET_stream = s
        elif name == 'MyMarkerStream3':
            MRK_stream = s

    if EEG_stream is None or ET_stream is None or MRK_stream is None:
        continue

    eeg = EEG_stream['time_series']
    eeg_t = EEG_stream['time_stamps']

    et = ET_stream['time_series']
    et_t = ET_stream['time_stamps']

    markers = np.array([m[0] for m in MRK_stream['time_series']])
    marker_t = MRK_stream['time_stamps']

    # Downsample
    eeg = eeg[::DOWNSAMPLE]
    eeg_t = eeg_t[::DOWNSAMPLE]

    # Align ET → EEG
    et_idx = np.searchsorted(et_t, eeg_t, side='left')
    et_idx[et_idx >= len(et)] = len(et) - 1
    et_aligned = et[et_idx]

    data = np.hstack([eeg, et_aligned])

    num_samples = (len(data) - window_size) // step_size
    if num_samples <= 0:
        continue

    samples = np.zeros((num_samples, window_size, data.shape[1]))
    labels = np.full(num_samples, -1)

    # Global marker encoding
    for m in np.unique(markers):
        if m not in marker_global_map:
            marker_global_map[m] = current_label_id
            current_label_id += 1

    # Assign labels
    for i in range(num_samples):
        start = i * step_size
        end = start + window_size

        t_start = eeg_t[start]
        t_end = eeg_t[end - 1]

        idx = np.where((marker_t >= t_start) & (marker_t <= t_end))[0]

        if len(idx) > 0:
            labels[i] = marker_global_map[markers[idx[0]]]

        samples[i] = data[start:end]

    valid = labels != -1
    all_samples.append(samples[valid])
    all_labels.append(labels[valid])

# Combine
X = np.concatenate(all_samples)
y = np.concatenate(all_labels)

print("Before fix → labels:", np.unique(y))

# ✅ FIX LABELS (CRITICAL)
unique_labels = np.unique(y)
label_map = {old: new for new, old in enumerate(unique_labels)}
y = np.array([label_map[val] for val in y])

print("After fix → labels:", np.unique(y))

# Normalize (important for RNN)
X = (X - X.mean()) / (X.std() + 1e-8)

np.save("/kaggle/working/X.npy", X)
np.save("/kaggle/working/y.npy", y)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np

# Load
X = np.load("/kaggle/working/X.npy")
y = np.load("/kaggle/working/y.npy")

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

print("Final label range:", y.min().item(), "to", y.max().item())

# Split
dataset = TensorDataset(X, y)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_ds, test_ds = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            nonlinearity='relu'
        )
        
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out)

input_size = X.shape[2]
num_classes = len(torch.unique(y))

model = RNNModel(input_size, 128, num_classes)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            nonlinearity='relu'
        )
        
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        return self.fc(out)

input_size = X.shape[2]
num_classes = len(torch.unique(y))

model = RNNModel(input_size, 128, num_classes)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)

        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)

        preds = model(xb)
        predicted = torch.argmax(preds, dim=1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print("Test Accuracy:", correct / total)

In [ ]:
unique_values = np.unique(y)

print("Unique values:", unique_values)
print("Number of unique classes:", len(unique_values))

In [ ]:
import pandas as pd

df = pd.DataFrame(X[0])  # first sample
print(df)

In [3]:
import scipy.io as sio
import os

# your dataset path
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

# list files to confirm names
print(os.listdir(base_path)[:10])

['S19.mat', 'S24.mat', 'processed_data.csv', 'S23.mat', 'S15.mat', 'S36.mat', 'S02.mat', 'S22.mat', 'S01.mat', 'S09.mat']


In [4]:
file_path = base_path + "/S01.mat"

data = sio.loadmat(file_path)

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['__header__', '__version__', '__globals__', 'S01'])


In [5]:
subject = data['S01']

print(type(subject))
print(subject.dtype)
print(subject.shape)

<class 'numpy.ndarray'>
[('Page1', 'O'), ('Page2', 'O'), ('Page3', 'O'), ('Page4', 'O'), ('Page5', 'O'), ('Page6', 'O'), ('Profile', 'O'), ('Demographics', 'O'), ('EEG_clean', 'O'), ('ET_clean', 'O')]
(1, 1)


In [6]:
page1 = subject['Page1'][0,0]

print(type(page1))
print(page1.dtype)
print(page1.dtype.names)

<class 'numpy.ndarray'>
[('Product1', 'O'), ('Product2', 'O'), ('Product3', 'O'), ('Product4', 'O'), ('Product5', 'O'), ('Product6', 'O'), ('Product7', 'O'), ('Product8', 'O'), ('Product9', 'O'), ('Product10', 'O'), ('Product11', 'O'), ('Product12', 'O'), ('Product13', 'O'), ('Product14', 'O'), ('Product15', 'O'), ('Product16', 'O'), ('Product17', 'O'), ('Product18', 'O'), ('Product19', 'O'), ('Product20', 'O'), ('Product21', 'O'), ('Product22', 'O'), ('Product23', 'O'), ('Product24', 'O')]
('Product1', 'Product2', 'Product3', 'Product4', 'Product5', 'Product6', 'Product7', 'Product8', 'Product9', 'Product10', 'Product11', 'Product12', 'Product13', 'Product14', 'Product15', 'Product16', 'Product17', 'Product18', 'Product19', 'Product20', 'Product21', 'Product22', 'Product23', 'Product24')


In [7]:
product1 = page1['Product1'][0,0]

print(type(product1))
print(product1.dtype)
print(product1.dtype.names)

<class 'numpy.ndarray'>
[('ProductInfo', 'O'), ('EEG_segments', 'O'), ('ET_segments', 'O')]
('ProductInfo', 'EEG_segments', 'ET_segments')


In [8]:
info = product1['ProductInfo'][0,0]

print(type(info))
print(info.dtype)
print(info.dtype.names)

<class 'numpy.ndarray'>
[('Bought', 'O'), ('Reasons', 'O'), ('Familiarity', 'O'), ('FrequentBuy', 'O'), ('Description', 'O')]
('Bought', 'Reasons', 'Familiarity', 'FrequentBuy', 'Description')


In [9]:
label = info['Bought'][0,0]

print(label)


[[0]]


In [10]:
label = int(info['Bought'][0,0][0,0])
print(label)

0


In [11]:
eeg = product1['EEG_segments'][0,0]

print(type(eeg))
print(eeg.shape)

<class 'numpy.ndarray'>
(8, 2)


In [12]:
eeg_flat = eeg.flatten()

print(eeg_flat)
print(len(eeg_flat))

[  130   398  9178  9180  9903  9905  9948  9950  9963  9965 10485 10488
 19860 19928 21920 22043]
16


In [13]:
et = product1['ET_segments'][0,0]

print(type(et))
print(et.shape)

<class 'numpy.ndarray'>
(8, 2)


In [14]:
et_flat = et.flatten()

print(et_flat)
print(len(et_flat))

[  54  161 3673 3674 3963 3964 3981 3982 3987 3988 4196 4197 7946 7973
 8770 8819]
16


In [15]:
import numpy as np

row = np.concatenate([eeg_flat, et_flat, [label]])

print(row)
print(len(row))

[  130   398  9178  9180  9903  9905  9948  9950  9963  9965 10485 10488
 19860 19928 21920 22043    54   161  3673  3674  3963  3964  3981  3982
  3987  3988  4196  4197  7946  7973  8770  8819     0]
33


In [16]:
import numpy as np
import pandas as pd

rows = []

# loop over all pages
for page_name in subject.dtype.names:
    
    # skip non-product fields
    if "Page" not in page_name:
        continue
    
    page = subject[page_name][0,0]
    
    # loop over products
    for product_name in page.dtype.names:
        
        product = page[product_name][0,0]
        
        # extract label
        info = product['ProductInfo'][0,0]
        label = int(info['Bought'][0,0][0,0])
        
        # extract EEG
        eeg = product['EEG_segments'][0,0]
        eeg_flat = eeg.flatten()
        
        # extract ET
        et = product['ET_segments'][0,0]
        et_flat = et.flatten()
        
        # combine
        row = np.concatenate([eeg_flat, et_flat, [label]])
        
        rows.append(row)

# create dataframe
df_s01 = pd.DataFrame(rows)

print(df_s01.shape)
df_s01.head()

(144, 117)


,0,1,2,3,4,5,6,7,8,9,...,107,108,109,110,111,112,113,114,115,116
0,130,398.0,9178.0,9180.0,9903.0,9905.0,9948.0,9950.0,9963.0,9965.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,125,128.0,400.0,595.0,660.0,663.0,9180.0,9258.0,9965.0,9968.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,595,950.0,8740.0,8743.0,19705.0,19708.0,19720.0,19768.0,19935.0,19938.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2593,2960.0,3163.0,3400.0,3405.0,3613.0,8913.0,8915.0,8938.0,9000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,120,123.0,2538.0,2590.0,2960.0,3160.0,3615.0,3620.0,4103.0,4253.0,...,8464.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
lengths = []

for row in rows:
    lengths.append(len(row))

print(set(lengths))

{1, 5, 9, 13, 17, 21, 25, 29, 33, 37, 41, 49, 53, 57, 61, 65, 69, 97, 109, 117}


In [18]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split

# -------------------------
# LOAD + BUILD DATASET
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences = []
lengths = []
labels = []

for file in os.listdir(base_path):
    
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    
    for page_name in subject.dtype.names:
        
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            
            product = page[product_name][0,0]
            
            # label
            info = product['ProductInfo'][0,0]
            label = int(info['Bought'][0,0][0,0])
            
            # EEG + ET
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            # -------------------------
            # FILTER BAD DATA
            # -------------------------
            if eeg.size == 0 or et.size == 0:
                continue
            
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)
            
            if seq.shape[0] == 0 or seq.shape[1] != 4:
                continue
            
            # store
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)

# -------------------------
# PAD SEQUENCES
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("min len:", lengths.min().item(), "max len:", lengths.max().item())

# -------------------------
# TRAIN / TEST SPLIT
# -------------------------
indices = np.arange(len(X))

train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]

# -------------------------
# RNN MODEL
# -------------------------
class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rnn = nn.RNN(
            input_size=4,
            hidden_size=32,
            batch_first=True
        )
        
        self.fc = nn.Linear(32, 1)
    
    def forward(self, x, lengths):
        
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        _, hidden = self.rnn(packed)
        
        out = self.fc(hidden[-1])
        return torch.sigmoid(out)

# -------------------------
# TRAINING
# -------------------------
model = RNNModel()

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    
    model.train()
    
    outputs = model(X_train, len_train)
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    preds = model(X_test, len_test)
    preds = (preds > 0.5).float()
    
    accuracy = (preds == y_test).float().mean()
    
print("Test Accuracy:", accuracy.item())

X shape: torch.Size([5931, 246, 4])
y shape: torch.Size([5931, 1])
min len: 1 max len: 246
Epoch 1, Loss: 0.6786
Epoch 2, Loss: 0.6621
Epoch 3, Loss: 0.6474
Epoch 4, Loss: 0.6338
Epoch 5, Loss: 0.6212
Epoch 6, Loss: 0.6087
Epoch 7, Loss: 0.5961
Epoch 8, Loss: 0.5816
Epoch 9, Loss: 0.5555
Epoch 10, Loss: 0.5466
Test Accuracy: 0.8668913245201111


In [19]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score

# -------------------------
# LOAD + BUILD DATASET
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences = []
lengths = []
labels = []
subject_ids = []

for file in os.listdir(base_path):
    
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    
    # get subject key (S01, S02...)
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            
            product = page[product_name][0,0]
            
            # label
            info = product['ProductInfo'][0,0]
            label = int(info['Bought'][0,0][0,0])
            
            # EEG + ET
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            # -------------------------
            # FILTER BAD DATA
            # -------------------------
            if eeg.size == 0 or et.size == 0:
                continue
            
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)  # (t,4)
            
            if seq.shape[0] == 0 or seq.shape[1] != 4:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD SEQUENCES
# -------------------------
X = pad_sequence(sequences, batch_first=True)  # (N, max_len, 4)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("min len:", lengths.min().item(), "max len:", lengths.max().item())

# -------------------------
# OPTIONAL: FILTER VERY SHORT SEQUENCES
# -------------------------
mask = lengths >= 3
X = X[mask]
lengths = lengths[mask]
y = y[mask]
subject_ids = np.array(subject_ids)[mask.numpy()]

print("After filtering short sequences:", X.shape)

# -------------------------
# SUBJECT-LEVEL SPLIT
# -------------------------
groups = np.array(subject_ids)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(np.zeros(len(groups)), groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]

print("Train samples:", len(X_train))
print("Test samples:", len(X_test))

# -------------------------
# RNN MODEL
# -------------------------
class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rnn = nn.RNN(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Linear(64, 1)
    
    def forward(self, x, lengths):
        
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        _, hidden = self.rnn(packed)
        
        out = self.fc(hidden[-1])
        return torch.sigmoid(out)

# -------------------------
# TRAINING
# -------------------------
model = RNNModel()

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    
    model.train()
    
    outputs = model(X_train, len_train)
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    probs = model(X_test, len_test).cpu().numpy().ravel()
    preds = (probs > 0.5).astype(int)
    y_true = y_test.cpu().numpy().ravel()

accuracy = (preds == y_true).mean()
roc_auc = roc_auc_score(y_true, probs)
f1 = f1_score(y_true, preds)

print("\n--- Evaluation ---")
print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)
print("F1 Score:", f1)

# -------------------------
# CLASS BALANCE CHECK
# -------------------------
print("\nPositive rate:", y.float().mean().item())

X shape: torch.Size([5931, 246, 4])
y shape: torch.Size([5931, 1])
min len: 1 max len: 246
After filtering short sequences: torch.Size([4267, 246, 4])
Train samples: 3303
Test samples: 964
Epoch 1, Loss: 0.7077
Epoch 2, Loss: 0.6038
Epoch 3, Loss: 0.5327
Epoch 4, Loss: 0.4818
Epoch 5, Loss: 0.4520
Epoch 6, Loss: 0.4363
Epoch 7, Loss: 0.4284
Epoch 8, Loss: 0.4270
Epoch 9, Loss: 0.4247
Epoch 10, Loss: 0.4302

--- Evaluation ---
Accuracy: 0.8381742738589212
ROC-AUC: 0.4845257362274689
F1 Score: 0.0

Positive rate: 0.1535036265850067


In [20]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score

# -------------------------
# LOAD + BUILD DATASET
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences = []
lengths = []
labels = []
subject_ids = []

for file in os.listdir(base_path):
    
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            
            product = page[product_name][0,0]
            
            # label
            info = product['ProductInfo'][0,0]
            label = int(info['Bought'][0,0][0,0])
            
            # EEG + ET
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            # FILTER BAD DATA
            if eeg.size == 0 or et.size == 0:
                continue
            
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)  # (t,4)
            
            if seq.shape[0] == 0 or seq.shape[1] != 4:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD SEQUENCES
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("min len:", lengths.min().item(), "max len:", lengths.max().item())

# -------------------------
# REMOVE VERY SHORT SEQUENCES
# -------------------------
mask = lengths >= 3
X = X[mask]
lengths = lengths[mask]
y = y[mask]
subject_ids = np.array(subject_ids)[mask.numpy()]

print("After filtering:", X.shape)

# -------------------------
# SUBJECT-LEVEL SPLIT
# -------------------------
groups = np.array(subject_ids)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(np.zeros(len(groups)), groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]

print("Train:", len(X_train), "Test:", len(X_test))

# -------------------------
# RNN MODEL (NO SIGMOID)
# -------------------------
class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rnn = nn.RNN(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Linear(64, 1)
    
    def forward(self, x, lengths):
        
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        _, hidden = self.rnn(packed)
        
        out = self.fc(hidden[-1])
        return out  # NO SIGMOID

# -------------------------
# TRAINING (IMBALANCE FIX)
# -------------------------
model = RNNModel()

# compute class weight
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    
    model.train()
    
    logits = model(X_train, len_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    logits = model(X_test, len_test)
    probs = torch.sigmoid(logits).cpu().numpy().ravel()
    
    # lower threshold (important)
    preds = (probs > 0.3).astype(int)
    y_true = y_test.cpu().numpy().ravel()

accuracy = (preds == y_true).mean()
roc_auc = roc_auc_score(y_true, probs)
f1 = f1_score(y_true, preds)

print("\n--- Evaluation ---")
print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)
print("F1 Score:", f1)

# -------------------------
# CLASS BALANCE
# -------------------------
print("\nPositive rate:", y.float().mean().item())

X shape: torch.Size([5931, 246, 4])
y shape: torch.Size([5931, 1])
min len: 1 max len: 246
After filtering: torch.Size([4267, 246, 4])
Train: 3303 Test: 964
Epoch 1, Loss: 1.2238
Epoch 2, Loss: 1.1888
Epoch 3, Loss: 1.1929
Epoch 4, Loss: 1.1951
Epoch 5, Loss: 1.1995
Epoch 6, Loss: 1.1801
Epoch 7, Loss: 1.1796
Epoch 8, Loss: 1.1871
Epoch 9, Loss: 1.1800
Epoch 10, Loss: 1.1781

--- Evaluation ---
Accuracy: 0.16182572614107885
ROC-AUC: 0.5897515232292461
F1 Score: 0.2785714285714286

Positive rate: 0.1535036265850067


In [19]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

# -------------------------
# LOAD DATA
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels, subject_ids = [], [], [], []

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            product = page[product_name][0,0]
            
            info = product['ProductInfo'][0,0]
            label = int(info['Bought'][0,0][0,0])
            
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            if eeg.size == 0 or et.size == 0:
                continue
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)
            
            if seq.shape[1] != 4 or seq.shape[0] == 0:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

# remove very short sequences
mask = lengths >= 3
X, lengths, y = X[mask], lengths[mask], y[mask]
subject_ids = np.array(subject_ids)[mask.numpy()]

print("Data shape:", X.shape)

# -------------------------
# SUBJECT SPLIT
# -------------------------
groups = np.array(subject_ids)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(np.zeros(len(groups)), groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]

# -------------------------
# NORMALIZATION (CRITICAL)
# -------------------------
mean = X_train.mean(dim=(0,1), keepdim=True)
std  = X_train.std(dim=(0,1), keepdim=True) + 1e-8

X_train = (X_train - mean) / std
X_test  = (X_test - mean) / std

# -------------------------
# MODEL (RNN)
# -------------------------
class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        self.fc = nn.Linear(64, 1)
    
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.rnn(packed)
        return self.fc(hidden[-1])  # no sigmoid

model = RNNModel()

# -------------------------
# LOSS (IMBALANCE FIX)
# -------------------------
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -------------------------
# TRAIN
# -------------------------
for epoch in range(10):
    model.train()
    
    logits = model(X_train, len_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    logits = model(X_test, len_test)
    probs = torch.sigmoid(logits).cpu().numpy().ravel()
    y_true = y_test.cpu().numpy().ravel()

# -------------------------
# FIND BEST THRESHOLD
# -------------------------
precision, recall, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("\nBest Threshold:", best_threshold)
print("Best F1:", f1_scores[best_idx])

# -------------------------
# FINAL METRICS
# -------------------------
preds = (probs > best_threshold).astype(int)

accuracy = (preds == y_true).mean()
roc_auc = roc_auc_score(y_true, probs)
f1 = f1_score(y_true, preds)

print("\n--- FINAL RESULTS ---")
print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)
print("F1 Score:", f1)

print("\nPositive rate:", y.float().mean().item())

Data shape: torch.Size([4267, 246, 4])
Epoch 1, Loss: 1.1809
Epoch 2, Loss: 1.1806
Epoch 3, Loss: 1.1768
Epoch 4, Loss: 1.1749
Epoch 5, Loss: 1.1751
Epoch 6, Loss: 1.1736
Epoch 7, Loss: 1.1750
Epoch 8, Loss: 1.1671
Epoch 9, Loss: 1.1701
Epoch 10, Loss: 1.1693

Best Threshold: 0.47198626
Best F1: 0.2970936462905471

--- FINAL RESULTS ---
Accuracy: 0.3215767634854772
ROC-AUC: 0.5650625158669713
F1 Score: 0.2952586206896552

Positive rate: 0.1535036265850067


In [20]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

# -------------------------
# LOAD DATA
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels, subject_ids = [], [], [], []

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            product = page[product_name][0,0]
            
            info = product['ProductInfo'][0,0]
            label = int(info['Bought'][0,0][0,0])
            
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            if eeg.size == 0 or et.size == 0:
                continue
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)
            
            if seq.shape[1] != 4 or seq.shape[0] == 0:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

# remove very short sequences
mask = lengths >= 3
X, lengths, y = X[mask], lengths[mask], y[mask]
subject_ids = np.array(subject_ids)[mask.numpy()]

print("Data shape:", X.shape)

# -------------------------
# SUBJECT SPLIT
# -------------------------
groups = np.array(subject_ids)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(np.zeros(len(groups)), groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]

# -------------------------
# NORMALIZATION (CRITICAL)
# -------------------------
mean = X_train.mean(dim=(0,1), keepdim=True)
std  = X_train.std(dim=(0,1), keepdim=True) + 1e-8

X_train = (X_train - mean) / std
X_test  = (X_test - mean) / std

# -------------------------
# MODEL (RNN)
# -------------------------
class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        self.fc = nn.Linear(64, 1)
    
    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.rnn(packed)
        return self.fc(hidden[-1])  # no sigmoid

model = RNNModel()

# -------------------------
# LOSS (IMBALANCE FIX)
# -------------------------
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -------------------------
# TRAIN
# -------------------------
for epoch in range(10):
    model.train()
    
    logits = model(X_train, len_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    logits = model(X_test, len_test)
    probs = torch.sigmoid(logits).cpu().numpy().ravel()
    y_true = y_test.cpu().numpy().ravel()

# -------------------------
# FIND BEST THRESHOLD
# -------------------------
precision, recall, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("\nBest Threshold:", best_threshold)
print("Best F1:", f1_scores[best_idx])

# -------------------------
# FINAL METRICS
# -------------------------
preds = (probs > best_threshold).astype(int)

accuracy = (preds == y_true).mean()
roc_auc = roc_auc_score(y_true, probs)
f1 = f1_score(y_true, preds)

print("\n--- FINAL RESULTS ---")
print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)
print("F1 Score:", f1)

print("\nPositive rate:", y.float().mean().item())

Data shape: torch.Size([4267, 246, 4])
Epoch 1, Loss: 1.2091
Epoch 2, Loss: 1.1810
Epoch 3, Loss: 1.1794
Epoch 4, Loss: 1.1837
Epoch 5, Loss: 1.1831
Epoch 6, Loss: 1.1817
Epoch 7, Loss: 1.1758
Epoch 8, Loss: 1.1735
Epoch 9, Loss: 1.1737
Epoch 10, Loss: 1.1730

Best Threshold: 0.45370907
Best F1: 0.300829872805909

--- FINAL RESULTS ---
Accuracy: 0.29979253112033194
ROC-AUC: 0.5547013835998985
F1 Score: 0.29906542056074764

Positive rate: 0.1535036265850067


In [23]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

# -------------------------
# LOAD DATA
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels, subject_ids = [], [], [], []

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            product = page[product_name][0,0]
            
            # label
            info = product['ProductInfo'][0,0]
            label = int(info['Bought'][0,0][0,0])
            
            # signals
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            # filtering
            if eeg.size == 0 or et.size == 0:
                continue
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)  # (t,4)
            
            if seq.shape[1] != 4 or seq.shape[0] == 0:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD SEQUENCES
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

print("Initial shape:", X.shape)

# -------------------------
# REMOVE VERY SHORT SEQUENCES
# -------------------------
mask = lengths >= 3
X, lengths, y = X[mask], lengths[mask], y[mask]
subject_ids = np.array(subject_ids)[mask.numpy()]

print("After filtering:", X.shape)

# -------------------------
# SUBJECT-LEVEL SPLIT
# -------------------------
groups = np.array(subject_ids)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(np.zeros(len(groups)), groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]

print("Train:", len(X_train), "Test:", len(X_test))

# -------------------------
# PER-SEQUENCE NORMALIZATION
# -------------------------
def normalize_per_sequence(X, lengths):
    Xn = X.clone()
    for i in range(X.size(0)):
        t = lengths[i]
        mu = X[i, :t].mean(dim=0, keepdim=True)
        sd = X[i, :t].std(dim=0, keepdim=True) + 1e-8
        Xn[i, :t] = (X[i, :t] - mu) / sd
    return Xn

X_train = normalize_per_sequence(X_train, len_train)
X_test  = normalize_per_sequence(X_test,  len_test)

# -------------------------
# GRU MODEL
# -------------------------
class GRUModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Linear(64, 1)
    
    def forward(self, x, lengths):
        
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        _, hidden = self.gru(packed)
        
        return self.fc(hidden[-1])  # NO sigmoid

model = GRUModel()

# -------------------------
# LOSS (IMBALANCE FIX)
# -------------------------
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# -------------------------
# TRAIN
# -------------------------
for epoch in range(25):
    
    model.train()
    
    logits = model(X_train, len_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    logits = model(X_test, len_test)
    probs = torch.sigmoid(logits).cpu().numpy().ravel()
    y_true = y_test.cpu().numpy().ravel()

# -------------------------
# BEST THRESHOLD
# -------------------------
precision, recall, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("\nBest Threshold:", best_threshold)
print("Best F1:", f1_scores[best_idx])

# -------------------------
# FINAL METRICS
# -------------------------
preds = (probs > best_threshold).astype(int)

accuracy = (preds == y_true).mean()
roc_auc = roc_auc_score(y_true, probs)
f1 = f1_score(y_true, preds)

print("\n--- FINAL RESULTS ---")
print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)
print("F1 Score:", f1)

print("\nPositive rate:", y.float().mean().item())

Initial shape: torch.Size([5931, 246, 4])
After filtering: torch.Size([4267, 246, 4])
Train: 3303 Test: 964
Epoch 1, Loss: 1.1797
Epoch 2, Loss: 1.1772
Epoch 3, Loss: 1.1750
Epoch 4, Loss: 1.1727
Epoch 5, Loss: 1.1706
Epoch 6, Loss: 1.1693
Epoch 7, Loss: 1.1681
Epoch 8, Loss: 1.1668
Epoch 9, Loss: 1.1658
Epoch 10, Loss: 1.1638
Epoch 11, Loss: 1.1641
Epoch 12, Loss: 1.1619
Epoch 13, Loss: 1.1613
Epoch 14, Loss: 1.1603
Epoch 15, Loss: 1.1586
Epoch 16, Loss: 1.1579
Epoch 17, Loss: 1.1563
Epoch 18, Loss: 1.1554
Epoch 19, Loss: 1.1550
Epoch 20, Loss: 1.1541
Epoch 21, Loss: 1.1528
Epoch 22, Loss: 1.1529
Epoch 23, Loss: 1.1524
Epoch 24, Loss: 1.1521
Epoch 25, Loss: 1.1525

Best Threshold: 0.48276335
Best F1: 0.3054187154010046

--- FINAL RESULTS ---
Accuracy: 0.5601659751037344
ROC-AUC: 0.5921712363544047
F1 Score: 0.3026315789473684

Positive rate: 0.1535036265850067


In [24]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

# -------------------------
# LOAD DATA
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels, subject_ids = [], [], [], []

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            product = page[product_name][0,0]
            
            info = product['ProductInfo'][0,0]
            label = int(info['Bought'][0,0][0,0])
            
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            if eeg.size == 0 or et.size == 0:
                continue
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)
            
            if seq.shape[1] != 4 or seq.shape[0] == 0:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

# remove short sequences
mask = lengths >= 3
X, lengths, y = X[mask], lengths[mask], y[mask]
subject_ids = np.array(subject_ids)[mask.numpy()]

print("Data:", X.shape)

# -------------------------
# SPLIT (subject-wise)
# -------------------------
groups = np.array(subject_ids)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(np.zeros(len(groups)), groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]

# -------------------------
# NORMALIZATION (per sequence)
# -------------------------
def normalize_per_sequence(X, lengths):
    Xn = X.clone()
    for i in range(X.size(0)):
        t = lengths[i]
        mu = X[i, :t].mean(dim=0, keepdim=True)
        sd = X[i, :t].std(dim=0, keepdim=True) + 1e-8
        Xn[i, :t] = (X[i, :t] - mu) / sd
    return Xn

X_train = normalize_per_sequence(X_train, len_train)
X_test  = normalize_per_sequence(X_test,  len_test)

# -------------------------
# ATTENTION GRU MODEL
# -------------------------
class AttentionGRU(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.attn = nn.Linear(64, 1)
        self.fc = nn.Linear(64, 1)

    def forward(self, x, lengths):
        
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        packed_out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        
        # mask padding
        max_len = out.size(1)
        mask = (torch.arange(max_len)[None, :].to(lengths.device) < lengths[:, None])
        
        # attention scores
        scores = self.attn(out).squeeze(-1)
        scores[~mask] = -1e9
        
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        
        context = (out * weights).sum(dim=1)
        
        return self.fc(context)

model = AttentionGRU()

# -------------------------
# LOSS (IMBALANCE FIX)
# -------------------------
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# -------------------------
# TRAIN
# -------------------------
for epoch in range(40):
    
    model.train()
    
    logits = model(X_train, len_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    logits = model(X_test, len_test)
    probs = torch.sigmoid(logits).cpu().numpy().ravel()
    y_true = y_test.cpu().numpy().ravel()

# -------------------------
# BEST THRESHOLD
# -------------------------
precision, recall, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("\nBest Threshold:", best_threshold)
print("Best F1:", f1_scores[best_idx])

# -------------------------
# FINAL METRICS
# -------------------------
preds = (probs > best_threshold).astype(int)

accuracy = (preds == y_true).mean()
roc_auc = roc_auc_score(y_true, probs)
f1 = f1_score(y_true, preds)

print("\n--- FINAL RESULTS ---")
print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)
print("F1 Score:", f1)

print("\nPositive rate:", y.float().mean().item())

Data: torch.Size([4267, 246, 4])
Epoch 1, Loss: 1.1763
Epoch 2, Loss: 1.1760
Epoch 3, Loss: 1.1754
Epoch 4, Loss: 1.1749
Epoch 5, Loss: 1.1746
Epoch 6, Loss: 1.1742
Epoch 7, Loss: 1.1735
Epoch 8, Loss: 1.1730
Epoch 9, Loss: 1.1729
Epoch 10, Loss: 1.1719
Epoch 11, Loss: 1.1718
Epoch 12, Loss: 1.1710
Epoch 13, Loss: 1.1709
Epoch 14, Loss: 1.1702
Epoch 15, Loss: 1.1694
Epoch 16, Loss: 1.1692
Epoch 17, Loss: 1.1682
Epoch 18, Loss: 1.1678
Epoch 19, Loss: 1.1670
Epoch 20, Loss: 1.1663
Epoch 21, Loss: 1.1648
Epoch 22, Loss: 1.1643
Epoch 23, Loss: 1.1632
Epoch 24, Loss: 1.1629
Epoch 25, Loss: 1.1613
Epoch 26, Loss: 1.1600
Epoch 27, Loss: 1.1593
Epoch 28, Loss: 1.1588
Epoch 29, Loss: 1.1586
Epoch 30, Loss: 1.1572
Epoch 31, Loss: 1.1569
Epoch 32, Loss: 1.1570
Epoch 33, Loss: 1.1571
Epoch 34, Loss: 1.1563
Epoch 35, Loss: 1.1564
Epoch 36, Loss: 1.1562
Epoch 37, Loss: 1.1563
Epoch 38, Loss: 1.1554
Epoch 39, Loss: 1.1544
Epoch 40, Loss: 1.1556

Best Threshold: 0.47450855
Best F1: 0.30990414961263263

In [25]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

# -------------------------
# LOAD DATA
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels, subject_ids = [], [], [], []

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            product = page[product_name][0,0]
            
            info = product['ProductInfo'][0,0]
            label = int(info['Bought'][0,0][0,0])
            
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            if eeg.size == 0 or et.size == 0:
                continue
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)
            
            if seq.shape[1] != 4 or seq.shape[0] == 0:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

# remove very short sequences
mask = lengths >= 3
X, lengths, y = X[mask], lengths[mask], y[mask]
subject_ids = np.array(subject_ids)[mask.numpy()]

print("Data shape:", X.shape)

# -------------------------
# SUBJECT SPLIT
# -------------------------
groups = np.array(subject_ids)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(np.zeros(len(groups)), groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]

print("Train:", len(X_train), "Test:", len(X_test))

# -------------------------
# NORMALIZATION
# -------------------------
def normalize_per_sequence(X, lengths):
    Xn = X.clone()
    for i in range(X.size(0)):
        t = lengths[i]
        mu = X[i, :t].mean(dim=0, keepdim=True)
        sd = X[i, :t].std(dim=0, keepdim=True) + 1e-8
        Xn[i, :t] = (X[i, :t] - mu) / sd
    return Xn

X_train = normalize_per_sequence(X_train, len_train)
X_test  = normalize_per_sequence(X_test,  len_test)

# -------------------------
# FUSION GRU MODEL
# -------------------------
class FusionGRU(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.eeg_gru = nn.GRU(
            input_size=2,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.et_gru = nn.GRU(
            input_size=2,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x, lengths):
        
        eeg = x[:, :, :2]
        et  = x[:, :, 2:]
        
        packed_eeg = nn.utils.rnn.pack_padded_sequence(
            eeg, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_et = nn.utils.rnn.pack_padded_sequence(
            et, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        _, h_eeg = self.eeg_gru(packed_eeg)
        _, h_et  = self.et_gru(packed_et)
        
        h_eeg = h_eeg[-1]
        h_et  = h_et[-1]
        
        combined = torch.cat([h_eeg, h_et], dim=1)
        
        return self.fc(combined)

model = FusionGRU()

# -------------------------
# LOSS (IMBALANCE)
# -------------------------
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# -------------------------
# TRAIN
# -------------------------
for epoch in range(30):
    
    model.train()
    
    logits = model(X_train, len_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    logits = model(X_test, len_test)
    probs = torch.sigmoid(logits).cpu().numpy().ravel()
    y_true = y_test.cpu().numpy().ravel()

# -------------------------
# BEST THRESHOLD
# -------------------------
precision, recall, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("\nBest Threshold:", best_threshold)
print("Best F1:", f1_scores[best_idx])

# -------------------------
# FINAL METRICS
# -------------------------
preds = (probs > best_threshold).astype(int)

accuracy = (preds == y_true).mean()
roc_auc = roc_auc_score(y_true, probs)
f1 = f1_score(y_true, preds)

print("\n--- FINAL RESULTS ---")
print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)
print("F1 Score:", f1)

print("\nPositive rate:", y.float().mean().item())

Data shape: torch.Size([4267, 246, 4])
Train: 3303 Test: 964
Epoch 1, Loss: 1.1761
Epoch 2, Loss: 1.1756
Epoch 3, Loss: 1.1746
Epoch 4, Loss: 1.1748
Epoch 5, Loss: 1.1728
Epoch 6, Loss: 1.1721
Epoch 7, Loss: 1.1727
Epoch 8, Loss: 1.1720
Epoch 9, Loss: 1.1718
Epoch 10, Loss: 1.1700
Epoch 11, Loss: 1.1710
Epoch 12, Loss: 1.1693
Epoch 13, Loss: 1.1689
Epoch 14, Loss: 1.1682
Epoch 15, Loss: 1.1675
Epoch 16, Loss: 1.1653
Epoch 17, Loss: 1.1662
Epoch 18, Loss: 1.1644
Epoch 19, Loss: 1.1641
Epoch 20, Loss: 1.1637
Epoch 21, Loss: 1.1626
Epoch 22, Loss: 1.1627
Epoch 23, Loss: 1.1629
Epoch 24, Loss: 1.1607
Epoch 25, Loss: 1.1625
Epoch 26, Loss: 1.1591
Epoch 27, Loss: 1.1572
Epoch 28, Loss: 1.1576
Epoch 29, Loss: 1.1574
Epoch 30, Loss: 1.1564

Best Threshold: 0.47821715
Best F1: 0.31057563257237786

--- FINAL RESULTS ---
Accuracy: 0.46473029045643155
ROC-AUC: 0.6009139375476008
F1 Score: 0.30831099195710454

Positive rate: 0.1535036265850067


In [2]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

# -------------------------
# DEVICE (GPU)
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------
# LOAD DATA
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels, subject_ids = [], [], [], []

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            product = page[product_name][0,0]
            
            info = product['ProductInfo'][0,0]
            label = int(info['Bought'][0,0][0,0])
            
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            if eeg.size == 0 or et.size == 0:
                continue
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)
            
            if seq.shape[1] != 4 or seq.shape[0] == 0:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

# remove short sequences
mask = lengths >= 3
X, lengths, y = X[mask], lengths[mask], y[mask]
subject_ids = np.array(subject_ids)[mask.numpy()]

print("Data shape:", X.shape)

# -------------------------
# SPLIT
# -------------------------
groups = np.array(subject_ids)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(np.zeros(len(groups)), groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]

# -------------------------
# NORMALIZATION
# -------------------------
def normalize_per_sequence(X, lengths):
    Xn = X.clone()
    for i in range(X.size(0)):
        t = lengths[i]
        mu = X[i, :t].mean(dim=0, keepdim=True)
        sd = X[i, :t].std(dim=0, keepdim=True) + 1e-8
        Xn[i, :t] = (X[i, :t] - mu) / sd
    return Xn

X_train = normalize_per_sequence(X_train, len_train)
X_test  = normalize_per_sequence(X_test,  len_test)

# move to GPU
X_train, X_test = X_train.to(device), X_test.to(device)
y_train, y_test = y_train.to(device), y_test.to(device)
len_train, len_test = len_train.to(device), len_test.to(device)

# -------------------------
# TRANSFORMER MODEL
# -------------------------
class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.input_proj = nn.Linear(4, 64)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=64,
            nhead=4,
            dim_feedforward=128,
            dropout=0.3,
            batch_first=True
        )
        
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1)
        )

    def forward(self, x, lengths):
        
        x = self.input_proj(x)
        
        max_len = x.size(1)
        mask = (torch.arange(max_len)[None, :].to(lengths.device) >= lengths[:, None])
        
        out = self.transformer(x, src_key_padding_mask=mask)
        
        # mean pooling (ignore padding)
        mask_inv = (~mask).unsqueeze(-1)
        out = (out * mask_inv).sum(dim=1) / mask_inv.sum(dim=1)
        
        return self.fc(out)

model = TransformerModel().to(device)

# -------------------------
# LOSS
# -------------------------
pos_weight = ((len(y_train) - y_train.sum()) / y_train.sum()).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

# -------------------------
# TRAIN
# -------------------------
for epoch in range(60):
    
    model.train()
    
    logits = model(X_train, len_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    logits = model(X_test, len_test)
    probs = torch.sigmoid(logits).cpu().numpy().ravel()
    y_true = y_test.cpu().numpy().ravel()

# -------------------------
# BEST THRESHOLD
# -------------------------
precision, recall, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("\nBest Threshold:", best_threshold)
print("Best F1:", f1_scores[best_idx])

# -------------------------
# FINAL METRICS
# -------------------------
preds = (probs > best_threshold).astype(int)

accuracy = (preds == y_true).mean()
roc_auc = roc_auc_score(y_true, probs)
f1 = f1_score(y_true, preds)

print("\n--- FINAL RESULTS ---")
print("Accuracy:", accuracy)
print("ROC-AUC:", roc_auc)
print("F1 Score:", f1)

print("\nPositive rate:", y.float().mean().item())

Using device: cuda
Data shape: torch.Size([4267, 246, 4])


AcceleratorError: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

device = torch.device("cpu")

# -------------------------
# LOAD DATA
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels, subject_ids = [], [], [], []

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            product = page[product_name][0,0]
            
            label = int(product['ProductInfo'][0,0]['Bought'][0,0][0,0])
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            if eeg.size == 0 or et.size == 0:
                continue
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)
            if seq.shape[0] < 3 or seq.shape[1] != 4:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
subject_ids = np.array(subject_ids)

# -------------------------
# STATISTICAL FEATURES
# -------------------------
def extract_features(X, lengths):
    feats = []
    for i in range(X.size(0)):
        t = lengths[i]
        seq = X[i, :t]
        
        eeg = seq[:, :2]
        et  = seq[:, 2:]
        
        f = torch.cat([
            eeg.mean(0), eeg.std(0), eeg.max(0).values, eeg.min(0).values,
            et.mean(0),  et.std(0),  et.max(0).values,  et.min(0).values
        ])
        feats.append(f)
    
    return torch.stack(feats)

X_feat = extract_features(X, lengths)

# -------------------------
# SPLIT (SUBJECT-LEVEL)
# -------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, groups=subject_ids))

X_train, X_test = X[train_idx], X[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
feat_train, feat_test = X_feat[train_idx], X_feat[test_idx]

# -------------------------
# NORMALIZATION
# -------------------------
def normalize_seq(X, lengths):
    Xn = X.clone()
    for i in range(X.size(0)):
        t = lengths[i]
        mu = X[i, :t].mean(0, keepdim=True)
        sd = X[i, :t].std(0, keepdim=True) + 1e-8
        Xn[i, :t] = (X[i, :t] - mu) / sd
    return Xn

X_train = normalize_seq(X_train, len_train)
X_test  = normalize_seq(X_test, len_test)

feat_mean = feat_train.mean(0)
feat_std  = feat_train.std(0) + 1e-8

feat_train = (feat_train - feat_mean) / feat_std
feat_test  = (feat_test - feat_mean) / feat_std

# -------------------------
# MODEL
# -------------------------
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Sequential(
            nn.Linear(64 + 16, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x, lengths, features):
        
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        
        _, hidden = self.gru(packed)
        seq_repr = hidden[-1]
        
        combined = torch.cat([seq_repr, features], dim=1)
        return self.fc(combined)

model = HybridModel()

# -------------------------
# LOSS
# -------------------------
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# -------------------------
# TRAIN
# -------------------------
for epoch in range(30):
    model.train()
    
    logits = model(X_train, len_train, feat_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    logits = model(X_test, len_test, feat_test)
    probs = torch.sigmoid(logits).numpy().ravel()
    y_true = y_test.numpy().ravel()

# best threshold
precision, recall, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

preds = (probs > best_threshold).astype(int)

print("\n--- FINAL RESULTS ---")
print("Accuracy:", (preds == y_true).mean())
print("ROC-AUC:", roc_auc_score(y_true, probs))
print("F1 Score:", f1_score(y_true, preds))

In [ ]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

device = torch.device("cpu")

# -------------------------
# LOAD DATA
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels, subject_ids = [], [], [], []

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            product = page[product_name][0,0]
            
            label = int(product['ProductInfo'][0,0]['Bought'][0,0][0,0])
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            if eeg.size == 0 or et.size == 0:
                continue
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)
            if seq.shape[0] < 3 or seq.shape[1] != 4:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
subject_ids = np.array(subject_ids)

# -------------------------
# ADVANCED FEATURE EXTRACTION
# -------------------------
def extract_features(X, lengths):
    feats = []
    
    for i in range(X.size(0)):
        t = lengths[i]
        seq = X[i, :t]
        
        eeg = seq[:, :2]
        et  = seq[:, 2:]
        
        # ---- basic stats ----
        stats = torch.cat([
            eeg.mean(0), eeg.std(0), eeg.max(0).values, eeg.min(0).values,
            et.mean(0),  et.std(0),  et.max(0).values,  et.min(0).values
        ])
        
        # ---- velocity (first difference) ----
        eeg_diff = torch.diff(eeg, dim=0)
        et_diff  = torch.diff(et, dim=0)
        
        vel = torch.cat([
            eeg_diff.mean(0), eeg_diff.std(0),
            et_diff.mean(0),  et_diff.std(0)
        ])
        
        # ---- energy ----
        energy = torch.cat([
            (eeg**2).mean(0),
            (et**2).mean(0)
        ])
        
        # ---- zero-crossing ----
        def zero_cross(x):
            return ((x[:-1] * x[1:]) < 0).float().mean(0)
        
        zc = torch.cat([
            zero_cross(eeg),
            zero_cross(et)
        ])
        
        features = torch.cat([stats, vel, energy, zc])
        feats.append(features)
    
    return torch.stack(feats)

X_feat = extract_features(X, lengths)

# -------------------------
# SUBJECT SPLIT
# -------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, groups=subject_ids))

X_train, X_test = X[train_idx], X[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
feat_train, feat_test = X_feat[train_idx], X_feat[test_idx]

# -------------------------
# NORMALIZATION
# -------------------------
def normalize_seq(X, lengths):
    Xn = X.clone()
    for i in range(X.size(0)):
        t = lengths[i]
        mu = X[i, :t].mean(0, keepdim=True)
        sd = X[i, :t].std(0, keepdim=True) + 1e-8
        Xn[i, :t] = (X[i, :t] - mu) / sd
    return Xn

X_train = normalize_seq(X_train, len_train)
X_test  = normalize_seq(X_test, len_test)

feat_mean = feat_train.mean(0)
feat_std  = feat_train.std(0) + 1e-8

feat_train = (feat_train - feat_mean) / feat_std
feat_test  = (feat_test - feat_mean) / feat_std

# -------------------------
# MODEL
# -------------------------
class HybridModel(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Sequential(
            nn.Linear(64 + feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x, lengths, features):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        seq_repr = hidden[-1]
        
        combined = torch.cat([seq_repr, features], dim=1)
        return self.fc(combined)

model = HybridModel(feat_train.shape[1]).to(device)

# -------------------------
# LOSS (IMBALANCE)
# -------------------------
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# -------------------------
# TRAIN
# -------------------------
for epoch in range(40):
    model.train()
    
    logits = model(X_train, len_train, feat_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    logits = model(X_test, len_test, feat_test)
    probs = torch.sigmoid(logits).numpy().ravel()
    y_true = y_test.numpy().ravel()

# best threshold
precision, recall, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

preds = (probs > best_threshold).astype(int)

print("\n--- FINAL RESULTS ---")
print("Accuracy:", (preds == y_true).mean())
print("ROC-AUC:", roc_auc_score(y_true, probs))
print("F1 Score:", f1_score(y_true, preds))

In [ ]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve

device = torch.device("cpu")

# -------------------------
# LOAD DATA
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels, subject_ids = [], [], [], []

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            product = page[product_name][0,0]
            
            label = int(product['ProductInfo'][0,0]['Bought'][0,0][0,0])
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            if eeg.size == 0 or et.size == 0:
                continue
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)
            
            if seq.shape[0] < 5 or seq.shape[1] != 4:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
subject_ids = np.array(subject_ids)

print("Dataset shape:", X.shape)

# -------------------------
# FEATURE EXTRACTION
# -------------------------
def extract_features(X, lengths):
    feats = []
    
    for i in range(X.size(0)):
        t = lengths[i]
        seq = X[i, :t]
        
        eeg = seq[:, :2]
        et  = seq[:, 2:]
        
        # ---- BASIC STATS ----
        stats = torch.cat([
            eeg.mean(0), eeg.std(0), eeg.max(0).values, eeg.min(0).values,
            et.mean(0),  et.std(0),  et.max(0).values,  et.min(0).values
        ])
        
        # ---- VELOCITY ----
        eeg_diff = torch.diff(eeg, dim=0)
        et_diff  = torch.diff(et, dim=0)
        
        vel = torch.cat([
            eeg_diff.mean(0), eeg_diff.std(0),
            et_diff.mean(0),  et_diff.std(0)
        ])
        
        # ---- ENERGY ----
        energy = torch.cat([
            (eeg**2).mean(0),
            (et**2).mean(0)
        ])
        
        # ---- ZERO CROSSING ----
        def zero_cross(x):
            if x.shape[0] < 2:
                return torch.zeros(x.shape[1])
            return ((x[:-1] * x[1:]) < 0).float().mean(0)
        
        zc = torch.cat([
            zero_cross(eeg),
            zero_cross(et)
        ])
        
        # ---- FFT FEATURES ----
        if eeg.shape[0] >= 8:  # ensure enough signal
            fft = torch.fft.rfft(eeg, dim=0)
            power = torch.abs(fft)**2
            
            n = power.shape[0]
            low  = power[:n//4].mean(0)
            mid  = power[n//4:n//2].mean(0)
            high = power[n//2:].mean(0)
        else:
            low = mid = high = torch.zeros(2)
        
        freq = torch.cat([low, mid, high])
        
        features = torch.cat([stats, vel, energy, zc, freq])
        feats.append(features)
    
    return torch.stack(feats)

X_feat = extract_features(X, lengths)

print("Feature shape:", X_feat.shape)

# -------------------------
# SPLIT
# -------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, groups=subject_ids))

X_train, X_test = X[train_idx], X[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
feat_train, feat_test = X_feat[train_idx], X_feat[test_idx]

# -------------------------
# NORMALIZATION
# -------------------------
def normalize_seq(X, lengths):
    Xn = X.clone()
    for i in range(X.size(0)):
        t = lengths[i]
        mu = X[i, :t].mean(0, keepdim=True)
        sd = X[i, :t].std(0, keepdim=True) + 1e-8
        Xn[i, :t] = (X[i, :t] - mu) / sd
    return Xn

X_train = normalize_seq(X_train, len_train)
X_test  = normalize_seq(X_test, len_test)

feat_mean = feat_train.mean(0)
feat_std  = feat_train.std(0) + 1e-8

feat_train = (feat_train - feat_mean) / feat_std
feat_test  = (feat_test - feat_mean) / feat_std

# -------------------------
# MODEL
# -------------------------
class HybridModel(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Sequential(
            nn.Linear(64 + feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x, lengths, features):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        seq_repr = hidden[-1]
        
        combined = torch.cat([seq_repr, features], dim=1)
        return self.fc(combined)

model = HybridModel(feat_train.shape[1])

# -------------------------
# LOSS
# -------------------------
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# -------------------------
# TRAIN
# -------------------------
for epoch in range(50):
    model.train()
    
    logits = model(X_train, len_train, feat_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# EVALUATION
# -------------------------
model.eval()

with torch.no_grad():
    logits = model(X_test, len_test, feat_test)
    probs = torch.sigmoid(logits).numpy().ravel()
    y_true = y_test.numpy().ravel()

precision, recall, thresholds = precision_recall_curve(y_true, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

preds = (probs > best_threshold).astype(int)

print("\n--- FINAL RESULTS ---")
print("Accuracy:", (preds == y_true).mean())
print("ROC-AUC:", roc_auc_score(y_true, probs))
print("F1 Score:", f1_score(y_true, preds))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score
from xgboost import XGBClassifier
import numpy as np

# -------------------------
# SPLIT (same as before)
# -------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_feat, groups=subject_ids))

X_train = X_feat[train_idx].numpy()
X_test  = X_feat[test_idx].numpy()

y_train = y[train_idx].numpy().ravel()
y_test  = y[test_idx].numpy().ravel()

# -------------------------
# MODEL
# -------------------------
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(len(y_train) - y_train.sum()) / y_train.sum(),
    eval_metric="logloss"
)

model.fit(X_train, y_train)

# -------------------------
# PREDICTION
# -------------------------
probs = model.predict_proba(X_test)[:, 1]

# threshold tuning
thresholds = np.linspace(0.1, 0.9, 50)
best_f1 = 0
best_threshold = 0.5

for t in thresholds:
    preds = (probs > t).astype(int)
    f1 = f1_score(y_test, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

preds = (probs > best_threshold).astype(int)

print("\n--- XGBoost RESULTS ---")
print("Accuracy:", (preds == y_test).mean())
print("ROC-AUC:", roc_auc_score(y_test, probs))
print("F1 Score:", f1_score(y_test, preds))
print("Best Threshold:", best_threshold)

In [1]:
import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, precision_recall_curve
from xgboost import XGBClassifier

device = torch.device("cpu")

# -------------------------
# LOAD DATA
# -------------------------
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels, subject_ids = [], [], [], []

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue
    
    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]
    subj_id = file.replace(".mat", "")
    
    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue
        
        page = subject[page_name][0,0]
        
        for product_name in page.dtype.names:
            product = page[product_name][0,0]
            
            label = int(product['ProductInfo'][0,0]['Bought'][0,0][0,0])
            eeg = product['EEG_segments'][0,0]
            et  = product['ET_segments'][0,0]
            
            if eeg.size == 0 or et.size == 0:
                continue
            if eeg.shape[0] != et.shape[0]:
                continue
            
            seq = np.concatenate([eeg, et], axis=1)
            
            if seq.shape[0] < 5 or seq.shape[1] != 4:
                continue
            
            sequences.append(torch.tensor(seq, dtype=torch.float32))
            lengths.append(seq.shape[0])
            labels.append(label)
            subject_ids.append(subj_id)

# -------------------------
# PAD
# -------------------------
X = pad_sequence(sequences, batch_first=True)
lengths = torch.tensor(lengths)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
subject_ids = np.array(subject_ids)

# -------------------------
# FEATURE EXTRACTION
# -------------------------
def extract_features(X, lengths):
    feats = []
    
    for i in range(X.size(0)):
        t = lengths[i]
        seq = X[i, :t]
        
        eeg = seq[:, :2]
        et  = seq[:, 2:]
        
        stats = torch.cat([
            eeg.mean(0), eeg.std(0), eeg.max(0).values, eeg.min(0).values,
            et.mean(0),  et.std(0),  et.max(0).values,  et.min(0).values
        ])
        
        eeg_diff = torch.diff(eeg, dim=0)
        et_diff  = torch.diff(et, dim=0)
        
        vel = torch.cat([
            eeg_diff.mean(0), eeg_diff.std(0),
            et_diff.mean(0),  et_diff.std(0)
        ])
        
        energy = torch.cat([
            (eeg**2).mean(0),
            (et**2).mean(0)
        ])
        
        def zero_cross(x):
            if x.shape[0] < 2:
                return torch.zeros(x.shape[1])
            return ((x[:-1] * x[1:]) < 0).float().mean(0)
        
        zc = torch.cat([
            zero_cross(eeg),
            zero_cross(et)
        ])
        
        features = torch.cat([stats, vel, energy, zc])
        feats.append(features)
    
    return torch.stack(feats)

X_feat = extract_features(X, lengths)

# -------------------------
# SPLIT
# -------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, groups=subject_ids))

X_train, X_test = X[train_idx], X[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
feat_train, feat_test = X_feat[train_idx], X_feat[test_idx]

# -------------------------
# NORMALIZATION
# -------------------------
def normalize_seq(X, lengths):
    Xn = X.clone()
    for i in range(X.size(0)):
        t = lengths[i]
        mu = X[i, :t].mean(0, keepdim=True)
        sd = X[i, :t].std(0, keepdim=True) + 1e-8
        Xn[i, :t] = (X[i, :t] - mu) / sd
    return Xn

X_train = normalize_seq(X_train, len_train)
X_test  = normalize_seq(X_test, len_test)

feat_mean = feat_train.mean(0)
feat_std  = feat_train.std(0) + 1e-8

feat_train = (feat_train - feat_mean) / feat_std
feat_test  = (feat_test - feat_mean) / feat_std

# -------------------------
# GRU MODEL
# -------------------------
class HybridModel(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=4,
            hidden_size=64,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )
        
        self.fc = nn.Sequential(
            nn.Linear(64 + feat_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x, lengths, features):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        seq_repr = hidden[-1]
        
        combined = torch.cat([seq_repr, features], dim=1)
        return self.fc(combined)

model = HybridModel(feat_train.shape[1])

# -------------------------
# TRAIN GRU
# -------------------------
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

for epoch in range(30):
    model.train()
    logits = model(X_train, len_train, feat_train)
    loss = criterion(logits, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"GRU Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -------------------------
# XGBOOST
# -------------------------
model_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=float((len(y_train) - y_train.sum()) / y_train.sum()),
    eval_metric="logloss"
)

model_xgb.fit(feat_train.numpy(), y_train.numpy().ravel())

# -------------------------
# ENSEMBLE
# -------------------------
model.eval()

with torch.no_grad():
    gru_logits = model(X_test, len_test, feat_test)
    gru_probs = torch.sigmoid(gru_logits).numpy().ravel()

xgb_probs = model_xgb.predict_proba(feat_test.numpy())[:, 1]

final_probs = 0.5 * gru_probs + 0.5 * xgb_probs

# -------------------------
# THRESHOLD TUNING
# -------------------------
thresholds = np.linspace(0.1, 0.9, 50)
best_f1 = 0
best_threshold = 0.5

y_true = y_test.numpy().ravel()

for t in thresholds:
    preds = (final_probs > t).astype(int)
    f1 = f1_score(y_true, preds)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

preds = (final_probs > best_threshold).astype(int)

print("\n--- FINAL ENSEMBLE RESULTS ---")
print("Accuracy:", (preds == y_true).mean())
print("ROC-AUC:", roc_auc_score(y_true, final_probs))
print("F1 Score:", f1_score(y_true, preds))
print("Best Threshold:", best_threshold)

GRU Epoch 1, Loss: 1.1586
GRU Epoch 2, Loss: 1.1592
GRU Epoch 3, Loss: 1.1557
GRU Epoch 4, Loss: 1.1526
GRU Epoch 5, Loss: 1.1527
GRU Epoch 6, Loss: 1.1520
GRU Epoch 7, Loss: 1.1495
GRU Epoch 8, Loss: 1.1492
GRU Epoch 9, Loss: 1.1491
GRU Epoch 10, Loss: 1.1481
GRU Epoch 11, Loss: 1.1446
GRU Epoch 12, Loss: 1.1426
GRU Epoch 13, Loss: 1.1394
GRU Epoch 14, Loss: 1.1374
GRU Epoch 15, Loss: 1.1396
GRU Epoch 16, Loss: 1.1421
GRU Epoch 17, Loss: 1.1386
GRU Epoch 18, Loss: 1.1375
GRU Epoch 19, Loss: 1.1348
GRU Epoch 20, Loss: 1.1358
GRU Epoch 21, Loss: 1.1326
GRU Epoch 22, Loss: 1.1354
GRU Epoch 23, Loss: 1.1357
GRU Epoch 24, Loss: 1.1324
GRU Epoch 25, Loss: 1.1382
GRU Epoch 26, Loss: 1.1315
GRU Epoch 27, Loss: 1.1355
GRU Epoch 28, Loss: 1.1307
GRU Epoch 29, Loss: 1.1291
GRU Epoch 30, Loss: 1.1322

--- FINAL ENSEMBLE RESULTS ---
Accuracy: 0.5644955300127714
ROC-AUC: 0.6238294390399128
F1 Score: 0.3504761904761905
Best Threshold: 0.4102040816326531


In [ ]:
 import os
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_recall_curve,
    accuracy_score, confusion_matrix, roc_curve, auc
)
from xgboost import XGBClassifier
import seaborn as sns
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# LOAD DATA
# =========================
base_path = "/kaggle/input/datasets/tanishguptttaaa/newdata"

sequences, lengths, labels = [], [], []

def extract_sequence(subject, product):
    try:
        eeg = subject['EEG_clean'][0,0]['Data'][0,0]
        et  = subject['ET_clean'][0,0]['Data'][0,0]
        segments = product['EEG_segments'][0,0]
    except:
        return None

    eeg = np.array(eeg, dtype=np.float32)
    et  = np.array(et, dtype=np.float32)

    scale = et.shape[1] / eeg.shape[1]

    seq_parts = []

    for seg in segments:
        start, end = int(seg[0]), int(seg[1])

        if end <= start or end > eeg.shape[1]:
            continue

        eeg_seg = eeg[:, start:end]

        et_start = int(start * scale)
        et_end   = int(end * scale)

        if et_end > et.shape[1]:
            continue

        et_seg = et[:, et_start:et_end]

        L = min(eeg_seg.shape[1], et_seg.shape[1])
        if L < 5:
            continue

        eeg_seg = eeg_seg[:, :L]
        et_seg  = et_seg[:, :L]

        combined = np.concatenate([eeg_seg, et_seg], axis=0)
        combined = np.nan_to_num(combined)

        mean = combined.mean(axis=1, keepdims=True)
        std = combined.std(axis=1, keepdims=True) + 1e-6
        combined = (combined - mean) / std

        seq_parts.append(combined.T)

    if len(seq_parts) == 0:
        return None

    return np.vstack(seq_parts)

for file in os.listdir(base_path):
    if not file.endswith(".mat"):
        continue

    data = sio.loadmat(os.path.join(base_path, file))
    key = [k for k in data.keys() if not k.startswith("__")][0]
    subject = data[key]

    for page_name in subject.dtype.names:
        if "Page" not in page_name:
            continue

        page = subject[page_name][0,0]

        for product_name in page.dtype.names:
            product = page[product_name][0,0]

            label = int(product['ProductInfo'][0,0]['Bought'][0,0][0,0])

            seq = extract_sequence(subject, product)
            if seq is None:
                continue

            sequences.append(seq)
            lengths.append(seq.shape[0])
            labels.append(label)

print("Loaded sequences:", len(sequences))

# =========================
# COMPRESS
# =========================
def compress(seq, target=200):
    if len(seq) <= target:
        return seq
    idx = np.linspace(0, len(seq)-1, target).astype(int)
    return seq[idx]

sequences = [compress(s) for s in sequences]
lengths = [len(s) for s in sequences]

# =========================
# TENSOR
# =========================
X = pad_sequence([torch.tensor(s) for s in sequences], batch_first=True)
y = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
lengths = torch.tensor(lengths)

# =========================
# FEATURE EXTRACTION
# =========================
def extract_features(X, lengths):
    feats = []
    for i in range(len(X)):
        seq = X[i, :lengths[i]].numpy()
        feats.append(np.concatenate([
            seq.mean(0), seq.std(0), seq.max(0), seq.min(0)
        ]))
    return np.array(feats)

X_feat = extract_features(X, lengths)

# =========================
# SPLIT
# =========================
indices = np.arange(len(X))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42)

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
len_train, len_test = lengths[train_idx], lengths[test_idx]

Xf_train, Xf_test = X_feat[train_idx], X_feat[test_idx]
yf_train, yf_test = y_train.numpy().ravel(), y_test.numpy().ravel()

# =========================
# DATASET
# =========================
class SeqDataset(Dataset):
    def __init__(self, X, y, lengths):
        self.X, self.y, self.lengths = X, y, lengths
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return self.X[i], self.y[i], self.lengths[i]

train_loader = DataLoader(SeqDataset(X_train, y_train, len_train), batch_size=64, shuffle=True)
test_loader  = DataLoader(SeqDataset(X_test, y_test, len_test), batch_size=64)

# =========================
# MODEL
# =========================
class Model(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.gru = nn.GRU(input_dim, 32, num_layers=2,
                          batch_first=True, bidirectional=True, dropout=0.3)
        self.attn = nn.Linear(64, 1)
        self.fc = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32, 1)
        )

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)

        w = torch.softmax(self.attn(out), dim=1)
        context = (out * w).sum(dim=1)

        return self.fc(context)

model = Model(X.shape[2]).to(device)

# =========================
# TRAIN + EARLY STOPPING
# =========================
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

train_loss_list = []
val_loss_list = []

best_val_loss = float('inf')
patience = 5
counter = 0

for epoch in range(50):

    model.train()
    total_train_loss = 0

    for xb, yb, lb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        logits = model(xb, lb)
        loss = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)
    train_loss_list.append(avg_train_loss)

    # VALIDATION
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for xb, yb, lb in test_loader:
            xb, yb = xb.to(device), yb.to(device)

            logits = model(xb, lb)
            loss = criterion(logits, yb)

            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(test_loader)
    val_loss_list.append(avg_val_loss)

    print(f"Epoch {epoch+1} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")

    # EARLY STOPPING
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered!")
            break

model.load_state_dict(torch.load("best_model.pth"))

# =========================
# LOSS CURVE
# =========================
plt.figure(figsize=(8,5))
plt.plot(train_loss_list, label="Train Loss")
plt.plot(val_loss_list, label="Val Loss")
plt.legend()
plt.title("Training vs Validation Loss")
plt.show()

# =========================
# EVALUATION
# =========================
model.eval()
probs = []

with torch.no_grad():
    for xb, _, lb in test_loader:
        xb = xb.to(device)
        out = torch.sigmoid(model(xb, lb)).cpu().numpy()
        probs.extend(out)

probs = np.array(probs).ravel()

# BEST THRESHOLD
precision, recall, thresholds = precision_recall_curve(yf_test, probs)
f1s = 2 * precision * recall / (precision + recall + 1e-8)
best_t = thresholds[np.argmax(f1s)]

# =========================
# XGBOOST
# =========================
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=(len(yf_train)-yf_train.sum())/yf_train.sum()
)
xgb.fit(Xf_train, yf_train)
probs_xgb = xgb.predict_proba(Xf_test)[:,1]

# FINAL ENSEMBLE
final_probs = 0.7 * probs + 0.3 * probs_xgb

print("\n--- FINAL RESULTS ---")
print("ROC-AUC:", roc_auc_score(yf_test, final_probs))
print("F1:", f1_score(yf_test, final_probs > best_t))
print("Accuracy:", accuracy_score(yf_test, final_probs > best_t))

# =========================
# CONFUSION MATRIX
# =========================
cm = confusion_matrix(yf_test, final_probs > best_t)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

# =========================
# ROC CURVE
# =========================
fpr, tpr, _ = roc_curve(yf_test, final_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, label=f"Final Model (AUC = {roc_auc:.3f})")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid()
plt.show()

# =========================
# ROC COMPARISON
# =========================
fpr_gru, tpr_gru, _ = roc_curve(yf_test, probs)
auc_gru = auc(fpr_gru, tpr_gru)

fpr_xgb, tpr_xgb, _ = roc_curve(yf_test, probs_xgb)
auc_xgb = auc(fpr_xgb, tpr_xgb)

plt.figure(figsize=(7,5))
plt.plot(fpr_gru, tpr_gru, label=f"GRU (AUC={auc_gru:.3f})")
plt.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={auc_xgb:.3f})")
plt.plot(fpr, tpr, label=f"Final (AUC={roc_auc:.3f})")
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Comparison")
plt.legend()
plt.grid()
plt.show()

Loaded sequences: 5541
